*Master IASD, PSL - 2022/2024 - O. Cappé (with code inputs from E. Kaufmann, A. Garivier and J. Achddou), 2024-03-21*

# Bandit Algorithms

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## Definition of the Bandit Environment

This part provides generic functions that can be instantiated to simulate and evaluate multi-armed bandit algorithms (for the time being with Gaussian and Bernoulli rewards). No need to edit these, unless you want to define other types of bandit models.

In [ ]:
class Environment:
    def __init__(self): pass

class Arm:
    def __init__(self): pass
    def draw(self): pass

class Bernoulli(Arm):
    """Bernoulli distributed arm."""
    def __init__(self, p):
        self.p = p
        self.expectation = p

    def draw(self):
        return float(np.random.random() < self.p)

class Gaussian(Arm):
    """Gaussian distributed arm."""
    def __init__(self, mu, sigma):
        self.sigma = sigma
        self.mu = mu
        self.expectation = mu

    def draw(self):
        return np.random.normal(self.mu, self.sigma)

class MAB(Environment):
    """Define a multi-armed bandit model with arms given in the 'arms' list."""
    def __init__(self, arms):
        self.arms = arms
        self.nbArms = len(arms)

    def play(self, policy, horizon):
        policy.reset()
        result = Result(self.nbArms, horizon)
        for t in range(horizon):
            choice = policy.choice()
            reward = self.arms[choice].draw()
            policy.update(choice, reward)
            index = None
            if hasattr(policy, "index"):
                index = policy.index
            result.store(t, choice, reward, index)
        return result

class Result:
    """Class for storing the outputs of a bandit experiment."""
    def __init__(self, nbArms, horizon):
        self.nbArms = nbArms
        self.choices = np.zeros(horizon, dtype=int)
        self.rewards = np.zeros(horizon, dtype=int)
        self.index = np.zeros((horizon, nbArms))
        self.means = np.zeros((horizon, nbArms))
        self.pulls = np.zeros((horizon, nbArms))

    def store(self, t, choice, reward, index):
        self.choices[t] = choice
        self.rewards[t] = reward
        self.index[t, :] = index
        self.pulls[t, :] = self.pulls[t-1, :]
        self.pulls[t, choice] += 1

    def getNbPulls(self, horizon):
        return self.pulls[horizon-1, :]

    def getRegret(self, bestExpectation):
        return np.cumsum(bestExpectation - self.rewards)

class Evaluation:
    """Class for computing statistics from multiple runs of a bandit experiment."""
    def __init__(self, env, pol, nbRepetitions, horizon, tsav=[]):
        if len(tsav) > 0:
            self.tsav = tsav
        else:
            self.tsav = np.arange(horizon)
        self.env = env
        self.pol = pol
        self.nbRepetitions = nbRepetitions
        self.horizon = horizon
        self.nbArms = env.nbArms
        self.nbPulls = np.zeros((self.nbRepetitions, self.nbArms))
        self.cumReward = np.zeros((self.nbRepetitions, len(tsav)))
        self.cumRegret = np.zeros((self.nbRepetitions, len(tsav)))
        print("Policy ", pol.__class__.__name__)
        for k in range(nbRepetitions):
            # if nbRepetitions < 10 or k % (nbRepetitions / 10) == 0:
            #    print("repetition ", k)
            result = env.play(pol, horizon)
            self.nbPulls[k, :] = result.getNbPulls(horizon)
            self.cumReward[k, :] = np.cumsum(result.rewards)[tsav]
            self.cumRegret[k, :] = (1 + self.tsav) * max([arm.expectation for arm in self.env.arms]) - self.cumReward[k,:]

    def meanReward(self):
        return sum(self.cumReward[:, -1]) / len(self.cumReward[:, -1])

    def cumRegret(self):
        self.cumReward

    def meanNbDraws(self):
        return np.mean(self.nbPulls, 0)

    def meanRegret(self):
        return (1 + self.tsav) * max([arm.expectation for arm in self.env.arms]) - np.mean(self.cumReward, 0)

## Utility functions for plotting regrets

These are generic functions that should not be modified.

In [ ]:
sns.set()
current_palette = sns.color_palette()
sns.set_style("ticks")
styles = ['o', '^', 's', 'D', 'p', 'v', '*', 'o']
colors = current_palette[0:12]

def plot_regret(env, policies, horizon, nbRep, styles, colors, title=False, ylim=None, std=True, name=''):
    """
    Plots the evolution of the average empirical regret of various policies in a fixed environment,
    using nbRep Monte Carlo trials.

    :param env: environment
    :type env: Environment object
    :param policies: list of policies to compare
    :type policies: list(Policy object)
    :param horizon: horizon
    :type horizon: int
    :param styles: list of styles
    :type styles: list
    :param colors: list of colors
    :type colors: list
    :param title: if True, will give a title to the plot
    :type title: Bool
    :param ylim: limit on the y axis
    :type ylim: float
    """
    k = 0
    plt.figure()
    tsav = np.arange(horizon)
    for policy in policies:
        plt.figure(1)
        ev = Evaluation(env, policy, nbRep, horizon,
                        tsav=np.arange(horizon))
        regret_list = ev.cumRegret
        meanRegret = regret_list.mean(axis=0)
        std_ar = regret_list.std(axis=0)
        plt.plot(1 + tsav, meanRegret,
                 color=colors[k], marker=styles[k], markevery=horizon / 20,
                 label=(policy.__class__.__name__))
        if std:
            plt.fill_between(1 + tsav, meanRegret - std_ar,
                             meanRegret + std_ar,
                             color=colors[k], alpha=0.2)
        plt.xlabel('Time')
        plt.ylabel('Regret')
        plt.legend()
        if ylim is not None:
            plt.ylim(top=ylim)
        k = k + 1

    if title is True:
        plt.title(name)
    plt.show()
    plt.close("all")

def plot_index(env, policy, horizon, rounds):
    arms_exp = [arm.expectation for arm in env.arms]
    result = env.play(policy, horizon)
    index = result.index
    fig, axs = plt.subplots(nrows=int(np.ceil(len(rounds) / 3)), ncols=3, sharex=True, #sharey=True,
                            figsize=(12, 2 * int(np.ceil(len(rounds) / 3))))
    for j, i in enumerate(rounds):
        ax = axs[int(j / 3)][j % 3]
        ax.scatter(range(env.nbArms), index[i, :], color=colors[0], label="Upper confidence bound")
        ax.scatter(range(env.nbArms), arms_exp, color=colors[2], marker='x', label="Unknown arm value")
        ax.legend()
        ax.set_ylim(top=1)
        ax.set_ylim(bottom=0)
        ax.set_title("UCBs at time " + str(i))
    plt.show()

def plot_pull(env, policy, horizon, rounds):
    result = env.play(policy, horizon)
    nbPulls = result.pulls
    arms_exp = [arm.expectation for arm in env.arms]
    opt_value = max(arms_exp)
    opt_arms = [arm for arm in range(env.nbArms) if env.arms[arm].expectation == opt_value]
    deltas = arms_exp - opt_value
    fig, axs = plt.subplots(nrows=int(np.ceil(len(rounds) / 3)), ncols=3, sharex=True,
                            figsize=(12, 2 * int(np.ceil(len(rounds) / 3))))
    for j, i in enumerate(rounds):
        ax = axs[int(j / 3)][j % 3]
        expected_opt_nbPulls = np.zeros(env.nbArms)
        for j in range(env.nbArms):
            if deltas[j] != 0:
                expected_opt_nbPulls[j] = np.log(i) / (2*deltas[j]**2)
            sum_subopt = expected_opt_nbPulls.sum()
            if deltas[j] == 0:
                expected_opt_nbPulls[j] = 1 / len(opt_arms) * (i - sum_subopt)
        ax.scatter(range(env.nbArms), nbPulls[i, :], label="actual nb of pulls")
        ax.scatter(range(env.nbArms), expected_opt_nbPulls, marker='x', label="theoretical baseline")
        ax.legend()
        ax.set_ylim(bottom=0)
        ax.set_title("Number of pulls of each arm at time " + str(i))
        ax.legend()
    plt.legend()
    plt.show()

## Definition of Policies

This is where bandit algorithms are defined, three examples are provided but **this is where you should add your own algorithms.** Try to implement at least one among the following policies for Bernoulli distributions:

- epsilon-greedy (easy modification of the Greedy policy)
- KLUCB (mostly requires a search function for finding a given level of the Bernoulli Kullback-Leibler divergence)
- Thompson sampling for Bernoulli distribution (requires to sample from the Beta posterior)

In [ ]:
class Policy:
    """Class that implements a generic policy template; expects (at least) the following functions to be defined."""
    def __init__(self): pass
    def reset(self, nbArms): pass
    def choice(self): pass
    def update(self): pass

In [ ]:
class Greedy(Policy):
    """The greedy (go with the empirical best) policy."""
    def __init__(self, nbArms):
        self.nbArms = nbArms
        self.nbDraws = np.ones(nbArms)
        self.cumReward = np.ones(nbArms)
        self.index = np.inf*np.ones(nbArms)
        self.bestArms = []

    def reset(self):
        self.t = 1
        for arm in range(self.nbArms):
            self.nbDraws[arm] = 0
            self.cumReward[arm] = 0.0
            self.index[arm] = np.inf
        self.bestArms = []

    def computeIndex(self, arm):
        if self.nbDraws[arm] < 1:
            return np.inf
        else:
            m = self.cumReward[arm]/self.nbDraws[arm] 
            return m 

    def update(self, arm, reward):
        self.nbDraws[arm] += 1
        self.cumReward[arm] += reward
        self.t += 1

    def choice (self):
        if self.t<self.nbArms:
            choice_ = int(self.t)%self.nbArms
            return choice_
        else:
            for arm in range(self.nbArms):
                self.index[arm] = self.computeIndex(arm)
                maxIndex = np.max (self.index)
                self.bestArms = [arm for arm in range(self.nbArms) if self.index[arm] == maxIndex]
            choice_ = int(np.random.choice(self.bestArms))
            return choice_

In [ ]:
class ETC(Policy):
    """The ETC (Explore Then Commit) policy."""
    def __init__(self, nbArms, expl):
        self.nbArms = nbArms
        self.nbDraws = np.ones(nbArms)
        self.cumReward = np.ones(nbArms)
        self.expl = expl
        self.index = np.inf*np.ones(nbArms)
        self.bestArms = []

    def reset(self):
        self.t = 1
        for arm in range(self.nbArms):
            self.nbDraws[arm] = 0
            self.cumReward[arm] = 0.0
            self.index[arm] = np.inf
        self.bestArms = []

    def computeIndex(self, arm):
        if self.nbDraws[arm] < 1:
            return np.inf
        else:
            m = self.cumReward[arm]/self.nbDraws[arm] 
            return m 

    def update(self, arm, reward):
        self.nbDraws[arm] += 1
        self.cumReward[arm] += reward
        self.t += 1

    def choice (self):
        if self.t<int(self.expl*self.nbArms):
            choice_ = int(self.t)%self.nbArms
            return choice_
        else:
            if self.t==int(self.expl*self.nbArms): 
                for arm in range(self.nbArms):
                    self.index[arm] = self.computeIndex(arm)
                maxIndex = np.max (self.index)
                self.bestArms = [arm for arm in range(self.nbArms) if self.index[arm] == maxIndex]
            choice_ = int(np.random.choice(self.bestArms))
            return choice_

In [ ]:
class UCB(Policy):
    """The UCB (Upper Confidence Bound) algorithm (for [0,1] variables using Hoeffding UCB)."""
    def __init__(self, nbArms):
        self.nbArms = nbArms
        self.nbDraws = np.ones(nbArms)
        self.cumReward = np.ones(nbArms)
        self.index = np.inf*np.ones(nbArms)

    def reset(self):
        self.t = 1
        for arm in range(self.nbArms):
            self.nbDraws[arm] = 0
            self.cumReward[arm] = 0.0
            self.index[arm] = np.inf

    def computeIndex(self, arm):
        if self.nbDraws[arm] < 1:
            return float('+infinity')
        else:
            m = self.cumReward[arm]/self.nbDraws[arm] 
            return m + np.sqrt(np.log(self.t)/ (2*self.nbDraws[arm]))

    def update(self, arm, reward):
        self.nbDraws[arm] += 1
        self.cumReward[arm] += reward
        self.t += 1

    def choice (self):
        if self.t<self.nbArms:
            choice_ = int(self.t)
            return choice_
        else :
            for arm in range(self.nbArms):
                self.index[arm] = self.computeIndex(arm)
            maxIndex = np.max (self.index)
            bestArms = [arm for arm in range(self.nbArms) if self.index[arm] == maxIndex]
            choice_ = int(np.random.choice(bestArms))
            return choice_

## Experiments

### Two Arms

Plot the regret of the greedy policy, the UCB policy, and the ETC policy for Bernoulli arms.

In [ ]:
means = np.array([0.5, 0.75])
env = MAB([Bernoulli(p) for p in means])
policy_UCB = UCB(env.nbArms)
policy_ETC = ETC(env.nbArms, 10)
greedy = Greedy(env.nbArms)
horizon = 1000
tsav = np.arange(horizon)
nbRep = 500

In [ ]:
policies = [policy_UCB, policy_ETC, greedy]
plot_regret(env, policies, horizon, nbRep, styles, colors, ylim = 100)

Same experiments when setting the exploration parameter to the optimal (horizon and parameter dependent) value for ETC.

In [ ]:
means = np.array([0.5, 0.75])
delta = 0.25
explOpt = np.ceil(np.log(horizon)/delta**2)
print("Number of exploration rounds: ", explOpt)
env = MAB([Bernoulli(p) for p in means])
policy_UCB = UCB(env.nbArms)
policy_ETC = ETC(env.nbArms, explOpt)
policy_greedy = Greedy(env.nbArms)
horizon = 1000
tsav = np.arange(horizon)
nbRep = 500

policies = [policy_UCB, policy_ETC, policy_greedy]
plot_regret(env, policies, horizon, nbRep, styles, colors, ylim = 100)

### What Happens When Adding a Third Arm?

In [ ]:
means = np.array([0.25, 0.5, 0.75])
env = MAB([Bernoulli(p) for p in means])
policy_UCB = UCB(env.nbArms)
policy_ETC = ETC(env.nbArms, explOpt)
policy_greedy = Greedy(env.nbArms)
horizon = 1000
tsav = np.arange(horizon)
nbRep = 500

policies = [policy_UCB, policy_ETC, policy_greedy]
plot_regret(env, policies, horizon, nbRep, styles, colors, ylim = 100)

#### Evolution of the Confidence Intervals for UCB 

In [ ]:
horizon = 600
rounds_end = list(np.arange(int(horizon/10), horizon, int(horizon/10)))
plot_index(env, policy_UCB, horizon, rounds_end)

#### Evolution of the Number of Pulls for UCB

In [ ]:
horizon = 600
rounds_end = list(np.arange(int(horizon/10), horizon, int(horizon/10)))
plot_pull(env, policy_UCB, horizon, rounds_end)